# Práctica 2 de PySpark: ChaskiFood

Segunda práctica de PySpark en Databricks Free Edition. En la primera trabajamos con tablas que ya venían limpias (`samples.nyctaxi`, `samples.tpch`). Ahora toca algo más parecido al trabajo real: llegan archivos crudos a un volumen y hay que convertirlos en tablas confiables para responder preguntas de negocio.

El caso: ChaskiFood es una app de delivery (ficticia) que opera en Lima, Arequipa y Trujillo. El equipo de datos recibe estos archivos:

| Archivo | Formato | Contenido |
|---|---|---|
| `restaurantes.csv` | CSV separado por `;` | restaurantes afiliados y su comisión |
| `clientes.csv` | CSV | clientes registrados, con bastante suciedad |
| `pedidos/pedidos_2025_MM.json` | JSON Lines, un archivo por mes | pedidos de enero a junio de 2025, con datos anidados |
| `actualizaciones/restaurantes_2025_07.csv` | CSV separado por `;` | cambios de restaurantes de julio |

Vamos a armar un pipeline por capas (arquitectura medallón):

```
raw (volumen)           bronze                   silver                     gold
restaurantes.csv  --->  bronze_restaurantes ---> silver_restaurantes --+
clientes.csv      --->  bronze_clientes     ---> silver_clientes     --+--> gold_ventas_diarias
pedidos/*.json    --->  bronze_pedidos      ---> silver_pedidos      --+    gold_segmentos_clientes
```

- bronze: los datos tal como llegan, más columnas de control (archivo de origen, fecha de ingesta)
- silver: datos limpios, sin duplicados y con los tipos correctos
- gold: tablas agregadas, listas para reportes

Se repasa todo lo de la práctica 1 (crear DataFrames, filtros, columnas nuevas, agregaciones, joins, ventanas, SQL, limpieza y escritura) y se sube un poco el nivel con:
- leer CSV y JSON con esquema explícito y detectar registros corruptos
- columnas anidadas (`struct` y `array`), `explode` y funciones de orden superior
- expresiones regulares y fechas que vienen en varios formatos
- `pivot`, `collect_set`, mediana y agregaciones condicionales
- joins `left_semi` y joins por rango
- ventanas con marcos móviles, porcentajes sobre el total y `ntile`
- SQL con CTE y consultas con parámetros
- Delta Lake: `MERGE INTO`, historial y time travel

Igual que antes, debajo de cada ejercicio hay una celda que comprueba el resultado: si imprime "Correcto", sigue; si sale un AssertionError, lee el mensaje y revisa tu código.

Antes de empezar:
1. Conecta el notebook a Serverless (selector de cómputo, arriba a la derecha).
2. Ejecuta las celdas en orden con Shift + Enter, porque casi todos los ejercicios usan resultados de los anteriores.
3. Si te pierdes, puedes volver a ejecutar todo desde el principio: la celda que genera los datos siempre produce exactamente los mismos archivos.

## 0. Preparación
Esta vez creamos un esquema propio, `workspace.chaskifood`, para no mezclar con lo de la práctica 1, y dentro de él un volumen `archivos` donde van a caer los archivos crudos.

Con `setCurrentCatalog` y `setCurrentDatabase` dejamos ese esquema como el actual, así podemos escribir `saveAsTable("silver_pedidos")` o `FROM silver_pedidos` sin poner el nombre completo. Si el notebook se desconecta, vuelve a ejecutar esta celda.

In [ ]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

CATALOGO = "workspace"
ESQUEMA = "chaskifood"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.archivos")
spark.catalog.setCurrentCatalog(CATALOGO)
spark.catalog.setCurrentDatabase(ESQUEMA)

RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/{ESQUEMA}/archivos"
RUTA_RAW = RUTA_VOLUMEN + "/raw"

def ok(mensaje="Correcto"):
    print(mensaje)

print("Versión de Spark:", spark.version)
print("Esquema actual:", spark.catalog.currentCatalog(), ".", spark.catalog.currentDatabase())

### Generar los archivos crudos
La siguiente celda escribe los archivos en `RUTA_RAW`. No hace falta entenderla (es Python normal, sin Spark): simula lo que nos mandarían los sistemas de la empresa, con los errores típicos que trae la data real. Usa una semilla fija, así que todos obtienen exactamente los mismos datos.

In [ ]:
# Genera los archivos crudos de ChaskiFood dentro del volumen.
# No hace falta entender esta celda: simula los archivos que nos mandarían los sistemas de la empresa.
# Usa una semilla fija, así que siempre produce exactamente los mismos datos.
import csv, json, math, os, random, unicodedata
from datetime import date, datetime, timedelta


def generar_datos_chaskifood(ruta, semilla=2025):
    rng = random.Random(semilla)
    for carpeta in ("", "/pedidos", "/actualizaciones"):
        os.makedirs(ruta + carpeta, exist_ok=True)

    def sin_tildes(texto):
        return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()

    def fecha_al_azar(desde, hasta):
        return desde + timedelta(days=rng.randint(0, (hasta - desde).days))

    def poisson(media):
        limite, k, p = math.exp(-media), 0, 1.0
        while True:
            p *= rng.random()
            if p <= limite:
                return k
            k += 1

    def escribir_csv(nombre, filas, sep=","):
        with open(f"{ruta}/{nombre}", "w", encoding="utf-8", newline="") as f:
            csv.writer(f, delimiter=sep).writerows(filas)

    def ensuciar(texto):
        r = rng.random()
        return texto.upper() if r < 0.15 else texto.lower() if r < 0.30 else f"  {texto} " if r < 0.40 else texto

    # ---------------- restaurantes ----------------
    MENUS = {
        "Pollería": [("1/4 de pollo a la brasa", 21.9), ("1/2 pollo a la brasa", 38.9), ("Pollo entero a la brasa", 69.9),
                     ("Porción de papas fritas", 9.9), ("Ensalada clásica", 8.5), ("Chicha morada 1L", 12.0)],
        "Chifa": [("Arroz chaufa", 24.9), ("Tallarín saltado", 26.9), ("Wantán frito", 14.9),
                  ("Chancho al tamarindo", 32.9), ("Kam lu wantán", 34.9), ("Inca Kola 1L", 8.0)],
        "Cevichería": [("Ceviche clásico", 35.9), ("Leche de tigre", 18.9), ("Jalea mixta", 42.9),
                       ("Arroz con mariscos", 38.9), ("Chicharrón de pescado", 33.9)],
        "Pizzería": [("Pizza americana familiar", 45.9), ("Pizza pepperoni familiar", 49.9),
                     ("Pizza hawaiana mediana", 34.9), ("Pan al ajo", 11.9), ("Gaseosa 1.5L", 10.0)],
        "Hamburguesas": [("Hamburguesa clásica", 19.9), ("Hamburguesa royal", 24.9), ("Hamburguesa a lo pobre", 27.9),
                         ("Papas fritas", 9.9), ("Milkshake de lúcuma", 14.9)],
        "Sushi": [("Acevichado roll", 32.9), ("Maki furai", 29.9), ("Nigiri mixto", 36.9), ("Gyozas", 18.9)],
        "Criolla": [("Lomo saltado", 36.9), ("Ají de gallina", 28.9), ("Arroz con pollo", 26.9),
                    ("Causa limeña", 19.9), ("Seco de res", 34.9), ("Chicha morada 1L", 12.0)],
        "Vegetariana": [("Bowl de quinua", 27.9), ("Hamburguesa de lentejas", 24.9), ("Ensalada andina", 22.9),
                        ("Jugo verde", 12.9)],
    }
    NOMBRES_REST = {
        "Pollería": ["Pollos Don Brasa", "La Leña Dorada", "El Gallo Fino"],
        "Chifa": ["Chifa Dragón de Oro", "Chifa Bambú", "Chifa Loto Azul"],
        "Cevichería": ["Mar Picante", "Ola Marina", "Caleta Sabrosa"],
        "Pizzería": ["Pizza Nostra", "Forno Andino", "La Masa Madre"],
        "Hamburguesas": ["Burger Chalaca", "Doble Queso", "La Parrilla Urbana"],
        "Sushi": ["Nikkei Roll", "Kaizen Sushi"],
        "Criolla": ["Sabor Criollo", "El Fogón Limeño", "La Sazón de Mamá Rosa"],
        "Vegetariana": ["Verde Quinua", "Raíz Bowl"],
    }
    DISTRITOS = {
        "Lima": ["Miraflores", "San Isidro", "Surco", "Barranco", "San Borja", "Jesús María", "Los Olivos",
                 "San Miguel", "La Molina"],
        "Arequipa": ["Cercado", "Yanahuara", "Cayma", "Paucarpata"],
        "Trujillo": ["Centro", "Víctor Larco", "La Esperanza"],
    }
    REPARTIDORES = {"Lima": ["R01", "R02", "R03", "R04", "R05", "R06"],
                    "Arequipa": ["R07", "R08", "R09"], "Trujillo": ["R10", "R11", "R12"]}

    categorias = list(MENUS)
    restaurantes = []
    for i, ciudad in enumerate(["Lima"] * 24 + ["Arequipa"] * 9 + ["Trujillo"] * 7, start=1):
        cat = categorias[(i - 1) % len(categorias)]
        restaurantes.append({"id": i, "nombre": rng.choice(NOMBRES_REST[cat]), "categoria": cat, "ciudad": ciudad,
                             "distrito": rng.choice(DISTRITOS[ciudad]), "comision": rng.choice([12, 15, 18, 20, 22.5]),
                             "fecha_alta": fecha_al_azar(date(2019, 1, 1), date(2024, 10, 1)),
                             "popularidad": rng.lognormvariate(0, 0.5)})

    def fila_restaurante(r):
        return [r["id"], r["nombre"] + ("  " if rng.random() < 0.2 else ""), r["categoria"], r["ciudad"],
                r["distrito"].lower() if rng.random() < 0.2 else r["distrito"],
                f"{r['comision']:g}%" if rng.random() < 0.8 else f"{r['comision']:g} %", r["fecha_alta"].isoformat()]

    encabezado_rest = ["id_restaurante", "nombre", "categoria", "ciudad", "distrito", "comision", "fecha_alta"]
    escribir_csv("restaurantes.csv", [encabezado_rest] + [fila_restaurante(r) for r in restaurantes], sep=";")

    # ---------------- clientes ----------------
    NOMBRES = ["María", "José", "Luis", "Carmen", "Juan", "Rosa", "Carlos", "Ana", "Jorge", "Lucía", "Miguel", "Sofía",
               "Diego", "Valeria", "Renzo", "Camila", "Andrés", "Daniela", "Piero", "Fiorella", "Kevin", "Milagros",
               "Bruno", "Ximena", "César", "Gabriela", "Raúl", "Patricia", "Óscar", "Karina"]
    APELLIDOS = ["Quispe", "Flores", "García", "Rodríguez", "Mamani", "Huamán", "Chávez", "Torres", "Rojas", "Vargas",
                 "Ramírez", "Castillo", "Mendoza", "Gutiérrez", "Sánchez", "Díaz", "Cárdenas", "Paredes", "Salazar",
                 "Espinoza"]
    clientes = []
    for i in range(1, 1501):
        nombre, apellido = rng.choice(NOMBRES), rng.choice(APELLIDOS)
        mes_inicio = rng.choices([1, 2, 3, 4, 5, 6], weights=[30, 18, 16, 14, 12, 10])[0] if rng.random() > 0.07 else None
        registro = fecha_al_azar(date(2023, 1, 1), date(2025, mes_inicio, 1) - timedelta(days=1) if mes_inicio else date(2025, 6, 15))
        clientes.append({
            "id": i, "nombre": f"{nombre} {apellido}", "mes_inicio": mes_inicio, "registro": registro,
            "ciudad": rng.choices(["Lima", "Arequipa", "Trujillo"], weights=[65, 20, 15])[0],
            "tasa": rng.lognormvariate(0.75, 0.6), "plan": "prime" if rng.random() < 0.25 else "free",
            "email": f"{sin_tildes(nombre).lower()}.{sin_tildes(apellido).lower()}{i}@"
                     + rng.choice(["gmail.com", "hotmail.com", "outlook.com", "yahoo.es"]),
            "telefono": "9" + "".join(str(rng.randint(0, 9)) for _ in range(8)),
            "actualizado": datetime.combine(registro, datetime.min.time()) + timedelta(minutes=rng.randint(0, 60 * 24 * 10)),
        })

    def fila_cliente(c, plan=None, email=None, actualizado=None):
        tel, r = c["telefono"], rng.random()
        tel = (tel if r < 0.40 else f"+51 {tel[:3]} {tel[3:6]} {tel[6:]}" if r < 0.65 else
               f"{tel[:3]}-{tel[3:6]}-{tel[6:]}" if r < 0.85 else f"(+51) {tel}" if r < 0.95 else
               None if r < 0.97 else "   " if r < 0.985 else tel[:5])
        email = email or c["email"]
        email = None if rng.random() < 0.02 else email.upper() if rng.random() < 0.12 else email
        if email and rng.random() < 0.10:
            email = f" {email}  "
        reg = c["registro"].isoformat() if rng.random() < 0.7 else c["registro"].strftime("%d/%m/%Y")
        return [c["id"], ensuciar(c["nombre"]), email, tel, None if rng.random() < 0.015 else ensuciar(c["ciudad"]),
                ensuciar(plan or c["plan"]), reg, (actualizado or c["actualizado"]).strftime("%Y-%m-%d %H:%M:%S")]

    # 25 clientes que cambiaron de plan: la versión vieja (free) también viene en el archivo
    actualizados = rng.sample([c for c in clientes if c["mes_inicio"]], 25)
    for c in actualizados:
        c["anterior"], c["plan"] = c["actualizado"], "prime"
        c["actualizado"] = c["anterior"] + timedelta(days=rng.randint(20, 120), minutes=rng.randint(0, 600))
    filas = {c["id"]: fila_cliente(c) for c in clientes}
    filas_clientes = list(filas.values())
    filas_clientes += [fila_cliente(c, plan="free", email=c["email"].split("@")[0] + "@yahoo.es", actualizado=c["anterior"])
                       for c in actualizados]
    ids_actualizados = [c["id"] for c in actualizados]
    candidatos = [c["id"] for c in clientes if c["id"] not in ids_actualizados]
    filas_clientes += [list(filas[i]) for i in rng.sample(candidatos, 60)]  # 60 copias exactas
    rng.shuffle(filas_clientes)
    escribir_csv("clientes.csv", [["id_cliente", "nombre", "email", "telefono", "ciudad", "plan", "fecha_registro",
                                   "fecha_actualizacion"]] + filas_clientes)

    # ---------------- pedidos ----------------
    rest_por_ciudad = {ciudad: [r for r in restaurantes if r["ciudad"] == ciudad] for ciudad in DISTRITOS}
    PESOS_HORA = [1, .5, .3, .2, .2, .3, .8, 1.5, 2, 2.5, 3, 5, 9, 10, 7, 4, 3, 4, 6, 9, 10, 8, 5, 2]
    PESOS_PAGO = {"Lima": [38, 32, 15, 15], "Arequipa": [35, 25, 12, 28], "Trujillo": [33, 22, 15, 30]}
    # el ceviche se pide más en verano (enero a marzo); el pollo y la comida criolla, cuando empieza el frío
    TEMPORADA = {"Cevichería": [2.2, 2.2, 1.8, 1.0, 0.7, 0.5], "Pollería": [0.8, 0.8, 0.9, 1.2, 1.5, 1.9],
                 "Criolla": [0.8, 0.8, 0.9, 1.1, 1.3, 1.5]}

    def momento_al_azar(mes):
        dias_mes = ((date(2025, mes + 1, 1) if mes < 12 else date(2026, 1, 1)) - date(2025, mes, 1)).days
        while True:
            dia = date(2025, mes, 1) + timedelta(days=rng.randint(0, dias_mes - 1))
            if dia.weekday() >= 4 or rng.random() < 0.75:  # de viernes a domingo hay más pedidos
                break
        return datetime(dia.year, dia.month, dia.day, rng.choices(range(24), weights=PESOS_HORA)[0],
                        rng.randint(0, 59), rng.randint(0, 59))

    def nuevo_pedido(id_cliente, ciudad, momento):
        opciones = rest_por_ciudad[ciudad] if rng.random() < 0.95 else restaurantes
        pesos = [x["popularidad"] * TEMPORADA.get(x["categoria"], [1] * 6)[momento.month - 1] for x in opciones]
        r = rng.choices(opciones, weights=pesos)[0]
        menu = MENUS[r["categoria"]]
        k = rng.choices([1, 2, 3, 4], weights=[45, 30, 17, 8])[0]
        items = [{"producto": p, "cantidad": rng.choices([1, 2, 3], weights=[70, 22, 8])[0], "precio_unitario": precio}
                 for p, precio in rng.sample(menu, min(k, len(menu)))]
        estado = rng.choices(["entregado", "cancelado", "reembolsado"], weights=[88, 9, 3])[0]
        distancia = round(min(12.0, max(0.3, rng.lognormvariate(1.0, 0.55))), 1)
        minutos = None if estado == "cancelado" else max(8, round(12 + distancia * 4.5 + rng.gauss(0, 6)))
        propina = 0.0 if estado == "cancelado" else float(rng.choices([0, 1, 2, 3, 5], weights=[55, 15, 15, 8, 7])[0])
        if rng.random() < 0.05:
            propina = None
        calificacion = None
        if estado == "entregado" and rng.random() < 0.72:
            base = 5 if minutos < 30 else 4 if minutos < 45 else 3
            calificacion = min(5, max(1, base + rng.choices([-2, -1, 0, 1], weights=[5, 20, 60, 15])[0]))
        return {
            "id_cliente": id_cliente, "id_restaurante": r["id"],
            "id_repartidor": None if estado == "cancelado" else rng.choice(REPARTIDORES[r["ciudad"]]),
            "fecha_hora": momento.strftime("%Y-%m-%dT%H:%M:%S"), "estado": estado,
            "metodo_pago": rng.choices(["yape", "tarjeta", "plin", "efectivo"], weights=PESOS_PAGO[r["ciudad"]])[0],
            "items": items, "entrega": {"distancia_km": distancia, "minutos": minutos, "propina": propina},
            "cupon": rng.choices(["CHASKI10", "BIENVENIDA20", "FINDE15", "YAPEA5", "VERANO2024"],
                                 weights=[35, 20, 20, 15, 10])[0] if rng.random() < 0.14 else None,
            "calificacion": calificacion,
        }

    pedidos = []
    for c in clientes:
        if not c["mes_inicio"]:
            continue
        for mes in range(c["mes_inicio"], 7):
            if mes > c["mes_inicio"] and rng.random() < 0.12:  # el cliente deja de pedir
                break
            n = max(1, poisson(c["tasa"])) if mes == c["mes_inicio"] else poisson(c["tasa"])
            for _ in range(n):
                momento = momento_al_azar(mes)
                pedidos.append((momento, nuevo_pedido(c["id"], c["ciudad"], momento)))
    for _ in range(80):  # pedidos de clientes que no están en clientes.csv
        momento = momento_al_azar(rng.randint(1, 6))
        pedidos.append((momento, nuevo_pedido(9000 + rng.randint(1, 40), rng.choice(list(DISTRITOS)), momento)))
    pedidos.sort(key=lambda x: x[0])

    lineas = {mes: [] for mes in range(1, 7)}
    for n, (momento, p) in enumerate(pedidos, start=1):
        lineas[momento.month].append(json.dumps({"id_pedido": f"CH-{n:06d}", **p}, ensure_ascii=False))
    originales = {mes: list(v) for mes, v in lineas.items()}
    for _ in range(150):  # pedidos reenviados: vuelven a aparecer en el archivo del mes siguiente
        mes = rng.randint(1, 5)
        lineas[mes + 1].append(rng.choice(originales[mes]))
    for k in range(5):  # líneas cortadas a la mitad
        mes = rng.randint(1, 6)
        texto = json.dumps({"id_pedido": f"CH-9{k:05d}", **nuevo_pedido(rng.randint(1, 1500), "Lima", momento_al_azar(mes))},
                           ensure_ascii=False)
        lineas[mes].insert(rng.randint(0, len(lineas[mes])), texto[: int(len(texto) * rng.uniform(0.4, 0.8))])
    for mes, contenido in lineas.items():
        with open(f"{ruta}/pedidos/pedidos_2025_{mes:02d}.json", "w", encoding="utf-8") as f:
            f.write("\n".join(contenido) + "\n")

    # ---------------- actualizaciones de julio ----------------
    cambios = [dict(r, comision=r["comision"] + rng.choice([-3, 2, 3])) for r in rng.sample(restaurantes, 4)]
    cambios[0]["nombre"] += " Express"
    for i, (cat, ciudad) in enumerate([("Sushi", "Lima"), ("Cevichería", "Arequipa")], start=41):
        cambios.append({"id": i, "nombre": rng.choice(NOMBRES_REST[cat]), "categoria": cat, "ciudad": ciudad,
                        "distrito": rng.choice(DISTRITOS[ciudad]), "comision": 15, "fecha_alta": date(2025, 7, 1)})
    escribir_csv("actualizaciones/restaurantes_2025_07.csv", [encabezado_rest] + [fila_restaurante(r) for r in cambios], sep=";")

    print(f"restaurantes.csv: {len(restaurantes)} filas")
    print(f"clientes.csv: {len(filas_clientes)} filas")
    print(f"pedidos/: {len(lineas)} archivos, {sum(len(v) for v in lineas.values())} líneas")
    print(f"actualizaciones/restaurantes_2025_07.csv: {len(cambios)} filas")


generar_datos_chaskifood(RUTA_RAW)

In [ ]:
display(dbutils.fs.ls(RUTA_RAW))

Antes de leer nada con Spark, conviene mirar los archivos tal cual. Los volúmenes se pueden leer también con Python normal (`open`), como cualquier carpeta:

In [ ]:
for archivo in ["restaurantes.csv", "clientes.csv", "pedidos/pedidos_2025_01.json"]:
    print(f"==> {archivo}")
    with open(f"{RUTA_RAW}/{archivo}", encoding="utf-8") as f:
        for _ in range(3):
            print(f.readline().rstrip())
    print()

Fíjate en lo que ya se ve: el CSV de restaurantes usa `;` como separador, en clientes hay mayúsculas, espacios de más y teléfonos con formatos distintos, y cada pedido trae una lista `items` y un objeto `entrega` anidados.

## Módulo 1. Repaso: DataFrames a mano

Hay dos tablas chicas que no vienen en archivos: los repartidores y los cupones vigentes. Las creamos con `spark.createDataFrame`. Recuerda que el esquema se puede declarar de dos formas:

```python
# con StructType
StructType([StructField("codigo", StringType(), True), StructField("valor", DoubleType(), True)])

# con un string DDL (más corto)
"codigo STRING, valor DOUBLE"
```

### Ejercicio 1.1: repartidores con `StructType`
Crea el DataFrame `repartidores` con los datos de abajo y un esquema explícito hecho con `StructType`:

| campo | tipo |
|---|---|
| `id_repartidor` | `StringType()` |
| `nombre` | `StringType()` |
| `vehiculo` | `StringType()` |
| `ciudad` | `StringType()` |
| `fecha_ingreso` | `DateType()` |

In [ ]:
datos_repartidores = [
    ("R01", "Wilmer Huanca",   "moto",      "Lima",     date(2023, 2, 14)),
    ("R02", "Katherine Soto",  "moto",      "Lima",     date(2023, 6, 1)),
    ("R03", "Jhon Ccori",      "bicicleta", "Lima",     date(2024, 1, 20)),
    ("R04", "Lizbeth Ramos",   "moto",      "Lima",     date(2022, 11, 3)),
    ("R05", "Erick Palomino",  "auto",      "Lima",     date(2024, 8, 12)),
    ("R06", "Yesenia Calle",   "bicicleta", "Lima",     date(2025, 1, 6)),
    ("R07", "Franco Apaza",    "moto",      "Arequipa", date(2023, 4, 9)),
    ("R08", "Nayeli Condori",  "moto",      "Arequipa", date(2024, 3, 15)),
    ("R09", "Alonso Zegarra",  "auto",      "Arequipa", date(2022, 9, 27)),
    ("R10", "Brenda Alvarado", "moto",      "Trujillo", date(2023, 7, 30)),
    ("R11", "Gianfranco Polo", "bicicleta", "Trujillo", date(2024, 5, 18)),
    ("R12", "Rocío Valverde",  "moto",      "Trujillo", date(2023, 12, 2)),
]


repartidores.printSchema()

In [ ]:
# comprobación
assert repartidores.count() == 12, "Deberías tener 12 repartidores"
assert repartidores.dtypes == [("id_repartidor", "string"), ("nombre", "string"), ("vehiculo", "string"),
                               ("ciudad", "string"), ("fecha_ingreso", "date")], f"Revisa nombres y tipos: {repartidores.dtypes}"
ok()

### Ejercicio 1.2: cupones con esquema DDL
Crea `cupones` con los datos de abajo, esta vez usando un esquema DDL (un string): `codigo STRING, tipo STRING, valor DOUBLE, minimo_compra DOUBLE`.

Cómo funcionan los cupones (lo vamos a necesitar en el módulo 4):
- `tipo = "porcentaje"`: el descuento es `valor` % del subtotal
- `tipo = "monto"`: el descuento es `valor` soles fijos
- en ambos casos, solo aplica si el subtotal del pedido es mayor o igual a `minimo_compra`

In [ ]:
datos_cupones = [
    ("CHASKI10",     "porcentaje", 10.0, 0.0),
    ("BIENVENIDA20", "porcentaje", 20.0, 30.0),
    ("FINDE15",      "porcentaje", 15.0, 40.0),
    ("YAPEA5",       "monto",       5.0, 25.0),
]


display(cupones)

In [ ]:
# comprobación
assert cupones.count() == 4
assert cupones.dtypes == [("codigo", "string"), ("tipo", "string"), ("valor", "double"), ("minimo_compra", "double")], \
    f"Revisa nombres y tipos: {cupones.dtypes}"
ok()

## Módulo 2. Ingesta: de archivos crudos a bronze

Para leer archivos usamos `spark.read` con opciones:

| Opción | Para qué sirve |
|---|---|
| `.option("header", True)` | la primera fila trae los nombres de las columnas |
| `.option("sep", ";")` | el separador no es una coma |
| `.schema("a INT, b STRING")` | esquema explícito |
| `.option("mode", "PERMISSIVE")` | (es el modo por defecto) si una línea no se puede leer, la fila queda con nulos en lugar de romper la lectura |
| `.option("columnNameOfCorruptRecord", "_corrupt_record")` | guarda el texto de la línea que no se pudo leer en esa columna (tiene que estar en el esquema) |

¿Por qué esquema explícito en lugar de `inferSchema`? Porque con `inferSchema` Spark lee los datos una vez más solo para adivinar los tipos, y a veces adivina mal: un teléfono termina como número, un código pierde los ceros a la izquierda, etc. En un pipeline real el esquema se declara.

Todo DataFrame leído de archivos tiene además una columna oculta, `_metadata`, con datos del archivo de origen (`_metadata.file_name`, `_metadata.file_path`, `_metadata.file_modification_time`...). No aparece en `select("*")`, hay que pedirla por nombre. En Unity Catalog reemplaza a la antigua función `input_file_name()`.

Sobre la escritura: Delta valida el esquema de la tabla (schema enforcement). Si una tabla ya existe con otras columnas, un `overwrite` falla con un error de "schema mismatch". Mientras practicamos vamos a agregar `.option("overwriteSchema", True)` para poder re-ejecutar las celdas sin problemas.

### Ejercicio 2.1: restaurantes (CSV con `;`)
Lee `RUTA_RAW + "/restaurantes.csv"` en `restaurantes_raw`, con encabezado, separador `;` y el esquema `ESQUEMA_RESTAURANTES`. La comisión viene como texto (por ejemplo `"18%"`), la limpiamos en el módulo 3.

In [ ]:
ESQUEMA_RESTAURANTES = "id_restaurante INT, nombre STRING, categoria STRING, ciudad STRING, distrito STRING, comision STRING, fecha_alta DATE"


display(restaurantes_raw)

In [ ]:
# comprobación
assert restaurantes_raw.columns == ["id_restaurante", "nombre", "categoria", "ciudad", "distrito", "comision", "fecha_alta"]
assert restaurantes_raw.count() == 40, "Deberían ser 40 restaurantes"
assert restaurantes_raw.filter(F.col("categoria").isNull()).count() == 0, "Hay categorías nulas: ¿pusiste sep=';'?"
ok()

### Ejercicio 2.2: clientes (CSV)
Lee `RUTA_RAW + "/clientes.csv"` en `clientes_raw`, con encabezado y el esquema `ESQUEMA_CLIENTES`. Casi todo se lee como texto a propósito: viene sucio y lo vamos a convertir después, ya limpio.

In [ ]:
ESQUEMA_CLIENTES = ("id_cliente INT, nombre STRING, email STRING, telefono STRING, ciudad STRING, "
                    "plan STRING, fecha_registro STRING, fecha_actualizacion STRING")


display(clientes_raw.limit(10))

In [ ]:
# comprobación
assert clientes_raw.columns == ["id_cliente", "nombre", "email", "telefono", "ciudad", "plan", "fecha_registro", "fecha_actualizacion"]
assert clientes_raw.count() == 1585, f"Esperaba 1585 filas, hay {clientes_raw.count()}"
ok()

### Ejercicio 2.3: pedidos (JSON anidado en varios archivos)
Los pedidos vienen en 6 archivos JSON Lines (un objeto JSON por línea) dentro de la carpeta `RUTA_RAW + "/pedidos"`. Si le pasas a Spark la carpeta, lee todos los archivos que tenga.

Cada pedido tiene campos anidados: `items` es una lista de productos (`ARRAY<STRUCT<...>>`) y `entrega` es un objeto (`STRUCT<...>`). El esquema ya está escrito en `ESQUEMA_PEDIDOS`, fíjate cómo se declaran esos tipos. Incluye la columna `_corrupt_record` para las líneas que no se puedan leer.

Crea `pedidos_raw`:
1. leyendo la carpeta con ese esquema y la opción `columnNameOfCorruptRecord`
2. agregando la columna `archivo` con `_metadata.file_name`
3. agregando la columna `fecha_ingesta` con `F.current_timestamp()`

Pista: `.select("*", F.col("_metadata.file_name").alias("archivo"))`

In [ ]:
ESQUEMA_PEDIDOS = """
    id_pedido STRING,
    id_cliente INT,
    id_restaurante INT,
    id_repartidor STRING,
    fecha_hora STRING,
    estado STRING,
    metodo_pago STRING,
    items ARRAY<STRUCT<producto: STRING, cantidad: INT, precio_unitario: DOUBLE>>,
    entrega STRUCT<distancia_km: DOUBLE, minutos: INT, propina: DOUBLE>,
    cupon STRING,
    calificacion INT,
    _corrupt_record STRING
"""


display(pedidos_raw.limit(5))

In [ ]:
# comprobación
assert pedidos_raw.count() == 11683, "Deberías leer todas las líneas de los 6 archivos (11683)"
assert {"archivo", "fecha_ingesta", "_corrupt_record"} <= set(pedidos_raw.columns)
assert pedidos_raw.select("archivo").distinct().count() == 6, "La columna archivo debería tener 6 valores distintos"
assert dict(pedidos_raw.dtypes)["items"].startswith("array<struct"), "Usa ESQUEMA_PEDIDOS"
ok()

### Ejercicio 2.4: guardar la capa bronze
Guarda los tres DataFrames como tablas Delta (modo `overwrite` y con `overwriteSchema`). Recorre el diccionario `tablas_bronze` con un `for` en vez de escribir tres veces lo mismo. Después lee la tabla `bronze_pedidos` en la variable `bronze_pedidos`.

In [ ]:
tablas_bronze = {
    "bronze_restaurantes": restaurantes_raw,
    "bronze_clientes": clientes_raw,
    "bronze_pedidos": pedidos_raw,
}


display(bronze_pedidos.limit(5))

In [ ]:
# comprobación
for nombre in tablas_bronze:
    assert spark.catalog.tableExists(f"{CATALOGO}.{ESQUEMA}.{nombre}"), f"No existe la tabla {nombre}"
assert bronze_pedidos.count() == 11683
ok("Correcto. Las tablas aparecen en Catalog > workspace > chaskifood")

### Ejercicio 2.5: revisar la ingesta
A partir de `bronze_pedidos`:
- guarda en `n_corruptos` cuántas filas tienen `_corrupt_record` no nulo
- crea `por_archivo` con el número de filas por `archivo` (columna `n_filas`), ordenado por archivo

Dato: si intentas el mismo conteo directamente sobre `pedidos_raw`, Spark puede rechazarlo con el error `QUERY_ONLY_CORRUPT_RECORD_COLUMN`, porque no permite consultar sobre el archivo crudo solo la columna `_corrupt_record`. Guardar primero en bronze evita el problema, y además es lo que se hace en un pipeline real: guardar lo que llegó, tal cual, antes de transformarlo.

In [ ]:
print(f"Líneas corruptas: {n_corruptos}")
display(por_archivo)
display(bronze_pedidos.filter(F.col("_corrupt_record").isNotNull()).select("archivo", "_corrupt_record"))

In [ ]:
# comprobación
assert n_corruptos == 5, f"Esperaba 5 líneas corruptas, hay {n_corruptos}"
assert por_archivo.columns == ["archivo", "n_filas"] and por_archivo.count() == 6
assert por_archivo.agg(F.sum("n_filas")).first()[0] == bronze_pedidos.count()
archivos = [r["archivo"] for r in por_archivo.collect()]
assert archivos == sorted(archivos), "Ordena por archivo"
ok()

## Módulo 3. Silver: limpiar clientes y restaurantes

Ahora pasamos de bronze a silver. En la práctica 1 usamos `trim`, `initcap`, `lower`, `fillna` y `try_cast`. Aquí se suman:

| Problema | Qué usar |
|---|---|
| quitar todo lo que no sea dígito | `F.regexp_replace("telefono", r"[^0-9]", "")` |
| extraer un patrón | `F.regexp_extract(col, r"(9\d{8})$", 1)` devuelve lo que calza con el grupo 1, o `""` si no hay coincidencia |
| convertir `""` en nulo | `F.nullif(col, F.lit(""))` |
| fechas en un formato conocido | `F.try_to_timestamp(col, F.lit("dd/MM/yyyy"))` devuelve `null` si el texto no calza con el formato |
| quedarse con el primer valor no nulo | `F.coalesce(a, b, ...)` |
| la última versión de cada registro | `row_number()` sobre una ventana particionada por la clave y ordenada por fecha descendente |

Con ANSI activo, `F.to_date("17/05/2024")` sin formato lanza un error. Por eso usamos `try_to_timestamp` con un formato explícito.

También vamos a empaquetar la limpieza en funciones. `df.transform(funcion)` aplica una función que recibe un DataFrame y devuelve otro. Así los pasos se pueden encadenar y reutilizar:

In [ ]:
def agregar_igv(df, tasa=0.18):
    return df.withColumn("precio_con_igv", F.round(F.col("precio") * (1 + tasa), 2))

productos = spark.createDataFrame([("Lomo saltado", 36.9), ("Ceviche clásico", 35.9)], "producto STRING, precio DOUBLE")
display(productos.transform(agregar_igv))
display(productos.transform(agregar_igv, tasa=0.10))   # los argumentos extra se pasan a la función

En este módulo trabajamos sobre las tablas bronze que guardaste en el módulo 2:

In [ ]:
bronze_clientes = spark.read.table("bronze_clientes")
bronze_restaurantes = spark.read.table("bronze_restaurantes")

### Ejercicio 3.1: perfil de calidad
Antes de limpiar conviene medir. Crea `perfil`, un DataFrame de una fila que tenga, para cada columna de texto de `bronze_clientes`, cuántos valores son nulos o están en blanco (solo espacios). Cada columna del resultado se llama igual que la columna original.

Pistas:
- las columnas de texto salen de `bronze_clientes.dtypes`: `[c for c, t in bronze_clientes.dtypes if t == "string"]`
- `(F.col(c).isNull() | (F.trim(c) == "")).cast("int")` vale 1 en los casos malos; súmalo con `F.sum` en una list comprehension

In [ ]:
display(perfil)

In [ ]:
# comprobación
fila = perfil.first()
assert "id_cliente" not in perfil.columns, "Solo columnas de texto"
assert len(perfil.columns) == 7
assert (fila["nombre"], fila["email"], fila["telefono"], fila["ciudad"]) == (0, 40, 60, 21), \
    f"Revisa el conteo (nulos o en blanco): {fila.asDict()}"
ok()

### Ejercicio 3.2: duplicados
Calcula:
- `n_duplicados_exactos`: filas de `bronze_clientes` menos filas distintas
- `n_ids_repetidos`: cuántos `id_cliente` aparecen más de una vez

¿Por qué no salen iguales? Porque algunos clientes aparecen dos veces con datos distintos (por ejemplo, alguien que pasó de free a prime). Esos no son duplicados exactos y `distinct()` no los elimina.

In [ ]:
print(n_duplicados_exactos, n_ids_repetidos)

In [ ]:
# comprobación
assert n_duplicados_exactos == 60, f"Esperaba 60 duplicados exactos, tienes {n_duplicados_exactos}"
assert n_ids_repetidos == 85, f"Esperaba 85 ids repetidos, tienes {n_ids_repetidos}"
ok()

### Ejercicio 3.3: normalizar textos
A partir de `bronze_clientes`, crea `clientes_norm`:
- `nombre`: sin espacios de sobra y con mayúscula inicial en cada palabra
- `email`: sin espacios y en minúsculas
- `ciudad`: sin espacios y con mayúscula inicial; los nulos pasan a `"Sin ciudad"`
- `plan`: sin espacios y en minúsculas

In [ ]:
display(clientes_norm.limit(10))

In [ ]:
# comprobación
ciudades = {r["ciudad"] for r in clientes_norm.select("ciudad").distinct().collect()}
planes = {r["plan"] for r in clientes_norm.select("plan").distinct().collect()}
assert ciudades == {"Lima", "Arequipa", "Trujillo", "Sin ciudad"}, f"Ciudades: {ciudades}"
assert planes == {"free", "prime"}, f"Planes: {planes}"
assert clientes_norm.filter(F.col("email") != F.lower(F.trim("email"))).count() == 0, "Revisa el email"
assert clientes_norm.filter(F.col("nombre") != F.initcap(F.trim("nombre"))).count() == 0, "Revisa el nombre"
assert clientes_norm.count() == bronze_clientes.count(), "Todavía no quites filas"
ok()

### Ejercicio 3.4: teléfonos con expresiones regulares
A partir de `clientes_norm`, crea `clientes_tel` reemplazando `telefono` por un número de 9 dígitos que empiece con 9:
1. quita todo lo que no sea dígito
2. quédate con los últimos 9 dígitos, si empiezan con 9: `F.regexp_extract(..., r"(9\d{8})$", 1)`
3. si no hubo coincidencia (`regexp_extract` devuelve `""`), el teléfono queda en `null`

| Viene así | Queda así |
|---|---|
| `"+51 987 654 321"` | `"987654321"` |
| `"987-654-321"` | `"987654321"` |
| `"(+51) 987654321"` | `"987654321"` |
| `"98765"` o `"   "` | `null` |

In [ ]:
display(clientes_tel.select("id_cliente", "telefono").limit(10))

In [ ]:
# comprobación
mal = clientes_tel.filter(F.col("telefono").isNotNull() & ~F.col("telefono").rlike(r"^9\d{8}$")).count()
assert mal == 0, f"{mal} teléfonos no tienen 9 dígitos empezando en 9"
nulos = clientes_tel.filter(F.col("telefono").isNull()).count()
assert nulos == 88, f"Esperaba 88 teléfonos nulos (vacíos o incompletos), tienes {nulos}"
ok()

### Ejercicio 3.5: fechas en dos formatos
A partir de `clientes_tel`, crea `clientes_fechas`:
- `fecha_registro` como `DATE`. Viene como `"2024-05-17"` o como `"17/05/2024"`: intenta cada formato con `F.try_to_timestamp` y quédate con el que funcione usando `F.coalesce`. Luego pásalo a fecha con `F.to_date`.
- `fecha_actualizacion` como `TIMESTAMP`. Viene como `"2025-03-01 14:20:00"`, que `F.to_timestamp` entiende sin indicar formato.

In [ ]:
display(clientes_fechas.select("id_cliente", "fecha_registro", "fecha_actualizacion").limit(10))

In [ ]:
# comprobación
tipos = dict(clientes_fechas.dtypes)
assert tipos["fecha_registro"] == "date" and tipos["fecha_actualizacion"] == "timestamp", f"Tipos: {tipos}"
assert clientes_fechas.filter(F.col("fecha_registro").isNull()).count() == 0, "Hay fechas que no se pudieron convertir"
r = clientes_fechas.agg(F.min("fecha_registro"), F.max("fecha_registro")).first()
assert r[0] >= date(2023, 1, 1) and r[1] <= date(2025, 6, 30), f"Rango raro: {r[0]} a {r[1]}. ¿Confundiste día y mes?"
ok()

### Ejercicio 3.6: quedarse con la versión más reciente
Crea `silver_clientes_df` con una sola fila por `id_cliente`: la de `fecha_actualizacion` más reciente. Usa una ventana con `row_number` y no dejes la columna auxiliar al final.

Después guárdalo como la tabla `silver_clientes` y léela de nuevo en la variable `silver_clientes`.

In [ ]:
display(silver_clientes.limit(10))

In [ ]:
# comprobación
assert silver_clientes.count() == 1500, f"Deberían quedar 1500 clientes, hay {silver_clientes.count()}"
assert silver_clientes.select("id_cliente").distinct().count() == 1500
assert silver_clientes.columns == bronze_clientes.columns, "No dejes columnas auxiliares"
ultima = clientes_fechas.groupBy("id_cliente").agg(F.max("fecha_actualizacion").alias("ultima"))
assert silver_clientes.join(ultima, "id_cliente").filter(F.col("fecha_actualizacion") != F.col("ultima")).count() == 0, \
    "Para algún cliente no te quedaste con la versión más reciente"
ok()

### Ejercicio 3.7: restaurantes con una función reutilizable
Escribe la función `limpiar_restaurantes(df)`. Recibe un DataFrame con las columnas de `bronze_restaurantes` y devuelve uno con:
- `nombre`: sin espacios de sobra
- `distrito`: sin espacios y con mayúscula inicial
- `comision_pct`: la comisión como número (`"18%"` pasa a `18.0` y `"22.5 %"` a `22.5`). Pista: borra todo lo que no sea dígito o punto y castea a `double`
- las columnas en este orden: `id_restaurante`, `nombre`, `categoria`, `ciudad`, `distrito`, `comision_pct`, `fecha_alta` (la columna `comision` original ya no va)

Aplícala con `bronze_restaurantes.transform(limpiar_restaurantes)` y guarda el resultado como la tabla `silver_restaurantes`. La vamos a reutilizar en el módulo 9.

In [ ]:
def limpiar_restaurantes(df):
    # tu código aquí
    return df


silver_restaurantes = spark.read.table("silver_restaurantes")
display(silver_restaurantes)

In [ ]:
# comprobación
columnas = ["id_restaurante", "nombre", "categoria", "ciudad", "distrito", "comision_pct", "fecha_alta"]
assert limpiar_restaurantes(bronze_restaurantes).columns == columnas, "Revisa las columnas que devuelve la función"
assert silver_restaurantes.columns == columnas and silver_restaurantes.count() == 40
assert dict(silver_restaurantes.dtypes)["comision_pct"] == "double"
assert silver_restaurantes.filter(~F.col("comision_pct").between(5, 30)).count() == 0, \
    "comision_pct debe ser un número como 18.0 (no 0.18 ni '18%')"
assert silver_restaurantes.filter((F.col("nombre") != F.trim("nombre")) | (F.col("distrito") != F.initcap("distrito"))).count() == 0
ok()

## Módulo 4. Silver: pedidos con datos anidados

Los pedidos traen dos tipos complejos:
- `entrega` es un `STRUCT`: un objeto con campos adentro
- `items` es un `ARRAY<STRUCT>`: una lista de objetos

| Para | Se usa |
|---|---|
| leer un campo de un struct | `F.col("entrega.minutos")` o `F.col("entrega")["minutos"]` |
| traer todos los campos de un struct | `df.select("entrega.*")` |
| cantidad de elementos de un array | `F.size("items")` |
| una fila por cada elemento del array | `F.explode("items")` (`F.explode_outer` si quieres conservar las filas con arrays vacíos o nulos) |
| operar dentro del array sin explotarlo | `F.transform`, `F.filter`, `F.aggregate`: funciones de orden superior, reciben una lambda |

In [ ]:
ejemplo = spark.createDataFrame(
    [("P1", [("Ceviche clásico", 2, 35.9), ("Chicha morada 1L", 1, 12.0)])],
    "id STRING, items ARRAY<STRUCT<producto: STRING, cantidad: INT, precio: DOUBLE>>",
)

# explode: una fila por item
display(ejemplo.select("id", F.explode("items").alias("item")).select("id", "item.*"))

# transform y filter: trabajan dentro del array, sin cambiar el número de filas
display(ejemplo.select(
    "id",
    F.transform("items", lambda x: F.upper(x["producto"])).alias("productos_mayus"),
    F.filter("items", lambda x: x["cantidad"] > 1).alias("items_con_varias_unidades"),
))

### Ejercicio 4.1: pedidos válidos y únicos
A partir de `bronze_pedidos`, crea `pedidos_validos`:
- sin las filas corruptas
- sin pedidos repetidos: algunos pedidos se reenviaron y aparecen otra vez en el archivo del mes siguiente
- sin la columna `_corrupt_record`

Ojo: `distinct()` no sirve aquí. Las filas repetidas no son idénticas porque la columna `archivo` cambia. Hay que quitar duplicados por `id_pedido`.

In [ ]:
print(pedidos_validos.count())

In [ ]:
# comprobación
assert "_corrupt_record" not in pedidos_validos.columns
assert pedidos_validos.count() == 11528, f"Deberían quedar 11528 pedidos, hay {pedidos_validos.count()}"
assert pedidos_validos.select("id_pedido").distinct().count() == pedidos_validos.count()
ok()

### Ejercicio 4.2: aplanar y tipar
A partir de `pedidos_validos`, crea `pedidos_planos` con estas columnas, en este orden:

| columna | de dónde sale |
|---|---|
| `id_pedido`, `id_cliente`, `id_restaurante`, `id_repartidor` | tal cual |
| `fecha_hora` | `fecha_hora` convertida a `TIMESTAMP` |
| `fecha` | la fecha (`DATE`) de `fecha_hora` |
| `hora` | la hora del día de `fecha_hora` |
| `anio_mes` | texto tipo `"2025-03"`, con `F.date_format(..., "yyyy-MM")` |
| `estado`, `metodo_pago`, `items` | tal cual |
| `n_items` | cantidad de elementos de `items` |
| `distancia_km` | `entrega.distancia_km` |
| `minutos_entrega` | `entrega.minutos` |
| `propina` | `entrega.propina`, con los nulos en 0 |
| `cupon`, `calificacion` | tal cual |

Las columnas `archivo` y `fecha_ingesta` eran de control de bronze, ya no van.

In [ ]:
display(pedidos_planos.limit(5))

In [ ]:
# comprobación
esperadas = ["id_pedido", "id_cliente", "id_restaurante", "id_repartidor", "fecha_hora", "fecha", "hora", "anio_mes",
             "estado", "metodo_pago", "items", "n_items", "distancia_km", "minutos_entrega", "propina", "cupon", "calificacion"]
assert pedidos_planos.columns == esperadas, f"Columnas: {pedidos_planos.columns}"
tipos = dict(pedidos_planos.dtypes)
assert tipos["fecha_hora"] == "timestamp" and tipos["fecha"] == "date" and tipos["n_items"] == "int", f"Tipos: {tipos}"
assert pedidos_planos.filter(F.col("propina").isNull()).count() == 0, "Rellena las propinas nulas con 0"
meses = [r["anio_mes"] for r in pedidos_planos.select("anio_mes").distinct().orderBy("anio_mes").collect()]
assert meses == ["2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"], f"Meses: {meses}"
ok()

### Ejercicio 4.3: una fila por producto (`explode`)
Crea `items_pedido` a partir de `pedidos_planos`, con una fila por cada producto de cada pedido y estas columnas:
`id_pedido`, `estado`, `id_restaurante`, `producto`, `cantidad`, `precio_unitario` y `subtotal_item` (= `cantidad * precio_unitario`, redondeado a 2).

In [ ]:
display(items_pedido.limit(10))

In [ ]:
# comprobación
assert items_pedido.columns == ["id_pedido", "estado", "id_restaurante", "producto", "cantidad", "precio_unitario", "subtotal_item"], \
    f"Columnas: {items_pedido.columns}"
assert items_pedido.count() == pedidos_planos.agg(F.sum("n_items")).first()[0], "Debe haber una fila por cada item"
ok()

### Ejercicio 4.4: subtotal sin explotar (`aggregate`)
Crea `pedidos_sub` agregando a `pedidos_planos` la columna `subtotal`: la suma de `cantidad * precio_unitario` de todos sus items, redondeada a 2.

Hazlo con `F.aggregate`, que recorre el array acumulando un valor, sin pasar por `explode` y `groupBy`:

```python
F.aggregate("items", F.lit(0.0), lambda acc, x: acc + x["cantidad"] * x["precio_unitario"])
```
El valor inicial es `0.0` (y no `0`) para que el acumulador sea `double` desde el principio.

In [ ]:
display(pedidos_sub.select("id_pedido", "items", "subtotal").limit(5))

In [ ]:
# comprobación
por_items = items_pedido.groupBy("id_pedido").agg(F.sum("subtotal_item").alias("suma_items"))
dif = pedidos_sub.join(por_items, "id_pedido").filter(F.abs(F.col("subtotal") - F.col("suma_items")) > 0.011).count()
assert "subtotal" in pedidos_sub.columns and dif == 0, f"{dif} pedidos con un subtotal distinto a la suma de sus items"
ok("Correcto. Mismo resultado que explode + groupBy, pero sin multiplicar filas")

### Ejercicio 4.5: aplicar cupones (left join + `when`)
A partir de `pedidos_sub`, crea `pedidos_total`:
1. Haz un left join con `F.broadcast(cupones)` donde `cupon == codigo`. Es left porque la mayoría de pedidos no tiene cupón, y algunos traen cupones que ya no existen (como `VERANO2024`).
2. Calcula `descuento`, redondeado a 2:
   - `0` si no hay cupón válido o si `subtotal < minimo_compra`
   - `subtotal * valor / 100` si `tipo == "porcentaje"`
   - `valor` si `tipo == "monto"`
3. Calcula `total = subtotal - descuento`, redondeado a 2.
4. Quita las columnas que vinieron de `cupones` (`codigo`, `tipo`, `valor`, `minimo_compra`).

In [ ]:
display(pedidos_total.filter(F.col("cupon").isNotNull()).select("id_pedido", "cupon", "subtotal", "descuento", "total").limit(10))

In [ ]:
# comprobación
assert pedidos_total.count() == pedidos_sub.count(), "El left join no debe perder ni duplicar pedidos"
assert set(pedidos_total.columns) == set(pedidos_sub.columns) | {"descuento", "total"}, "Quita las columnas de cupones"
assert pedidos_total.filter(F.col("descuento").isNull() | F.col("total").isNull()).count() == 0, "No debería haber nulos"
assert pedidos_total.filter((F.col("cupon") == "VERANO2024") & (F.col("descuento") != 0)).count() == 0, "VERANO2024 no existe"
assert pedidos_total.filter((F.col("cupon") == "FINDE15") & (F.col("subtotal") < 40) & (F.col("descuento") != 0)).count() == 0, \
    "Revisa el mínimo de compra"
assert pedidos_total.filter((F.col("cupon") == "YAPEA5") & (F.col("subtotal") >= 25) & (F.col("descuento") != 5)).count() == 0, \
    "Revisa los cupones de tipo monto"
assert pedidos_total.filter((F.col("cupon") == "BIENVENIDA20") & (F.col("subtotal") >= 30) &
                            (F.abs(F.col("descuento") - F.col("subtotal") * 0.2) > 0.011)).count() == 0, \
    "Revisa los cupones de tipo porcentaje"
assert pedidos_total.filter(F.abs(F.col("total") - (F.col("subtotal") - F.col("descuento"))) > 0.011).count() == 0
ok()

### Ejercicio 4.6: guardar `silver_pedidos`
Guarda `pedidos_total` como la tabla `silver_pedidos` y léela de nuevo en la variable `silver_pedidos`.

In [ ]:
display(silver_pedidos.limit(5))

In [ ]:
# comprobación
assert silver_pedidos.count() == 11528
assert {"subtotal", "descuento", "total", "items"} <= set(silver_pedidos.columns)
ok()

Hasta el 4.5 no se ejecutó nada: cada paso solo agregaba transformaciones al plan (evaluación perezosa). Recién el `saveAsTable` del 4.6 disparó todo. Con `explain()` puedes ver el plan físico que armó Spark; busca el `BroadcastHashJoin` del cruce con cupones:

In [ ]:
pedidos_total.explain()

## Módulo 5. Joins con más nivel

Repaso de la práctica 1 y dos tipos nuevos:

| `how=` | Qué devuelve |
|---|---|
| `"inner"` | solo las filas con pareja en ambos lados |
| `"left"` | todas las filas de la izquierda, con `null` donde no hay pareja |
| `"left_anti"` | las filas de la izquierda que no tienen pareja |
| `"left_semi"` (nuevo) | las filas de la izquierda que sí tienen pareja, solo con las columnas de la izquierda y sin duplicarlas (como un `WHERE EXISTS` en SQL) |

Join por rango (nuevo): la condición no tiene que ser una igualdad. Por ejemplo, para asignar a cada pedido el tramo de distancia que le corresponde:

```python
df.join(tramos, (df.km >= tramos.desde) & (df.km < tramos.hasta))
```

Cuidado con los nombres repetidos: `silver_clientes` y `silver_restaurantes` tienen las dos columnas `nombre` y `ciudad`. Si las unes sin más, el resultado tiene dos columnas `ciudad` y Spark se queja de ambigüedad cuando intentas usarlas. Lo más limpio es renombrar antes del join, con `select` + `alias` o con `withColumnRenamed`.

### Ejercicio 5.1: pedidos huérfanos
Algunos pedidos tienen un `id_cliente` que no existe en `silver_clientes` (un problema de integridad que hay que reportar). Crea `huerfanos` con esos pedidos usando `left_anti` y guarda cuántos son en `n_huerfanos`.

In [ ]:
print(f"{n_huerfanos} pedidos de clientes que no existen")
display(huerfanos.groupBy("id_cliente").count().orderBy("id_cliente"))

In [ ]:
# comprobación
assert n_huerfanos == 80, f"Esperaba 80 pedidos huérfanos, tienes {n_huerfanos}"
assert set(huerfanos.columns) == set(silver_pedidos.columns), "left_anti solo devuelve las columnas de la izquierda"
ok()

### Ejercicio 5.2: clientes activos e inactivos
Crea:
- `clientes_activos`: los clientes de `silver_clientes` que hicieron al menos un pedido (usa `left_semi`)
- `clientes_inactivos`: los que nunca pidieron (usa `left_anti`)

In [ ]:
print(f"Activos: {clientes_activos.count()}  |  Sin pedidos: {clientes_inactivos.count()}")

In [ ]:
# comprobación
assert clientes_activos.columns == silver_clientes.columns, "left_semi solo devuelve las columnas de la izquierda"
assert clientes_activos.count() == silver_pedidos.join(silver_clientes, "id_cliente").select("id_cliente").distinct().count(), \
    "left_semi no repite clientes aunque tengan muchos pedidos"
assert clientes_activos.count() + clientes_inactivos.count() == silver_clientes.count()
ok()

### Ejercicio 5.3: la tabla enriquecida
Crea `pedidos_enriquecidos` con todas las columnas de `silver_pedidos` más estas:

| columna nueva | viene de |
|---|---|
| `restaurante` | `silver_restaurantes.nombre` |
| `categoria`, `ciudad`, `comision_pct` | `silver_restaurantes` (la ciudad del restaurante) |
| `plan` | `silver_clientes.plan` |
| `ciudad_cliente` | `silver_clientes.ciudad` |

- Con restaurantes haz un inner join usando `F.broadcast` (es una tabla chica).
- Con clientes haz un left join, para no perder los pedidos huérfanos (van a quedar con `plan` nulo).
- Renombra antes de unir para que no haya columnas repetidas.

Este va a ser el DataFrame principal de los próximos módulos.

In [ ]:
display(pedidos_enriquecidos.select("id_pedido", "restaurante", "categoria", "ciudad", "plan", "ciudad_cliente", "total").limit(10))

In [ ]:
# comprobación
assert pedidos_enriquecidos.count() == silver_pedidos.count(), "No deberías perder ni duplicar pedidos"
assert len(pedidos_enriquecidos.columns) == len(set(pedidos_enriquecidos.columns)), "Hay columnas repetidas: renombra antes del join"
assert {"restaurante", "categoria", "ciudad", "comision_pct", "plan", "ciudad_cliente"} <= set(pedidos_enriquecidos.columns)
assert set(silver_pedidos.columns) <= set(pedidos_enriquecidos.columns)
assert pedidos_enriquecidos.filter(F.col("plan").isNull()).count() == 80, "Los pedidos huérfanos deben quedar con plan nulo (left join)"
ok()

### Ejercicio 5.4: costo de envío con un join por rango
El costo de envío depende de la distancia, según la tabla `tarifas_envio`: un pedido de 3.4 km cae en el tramo `[2, 5)` y paga 5 soles.

1. Crea `pedidos_envio` uniendo `pedidos_enriquecidos` con `tarifas_envio` donde `km_desde <= distancia_km < km_hasta`. Quita `km_desde` y `km_hasta` del resultado.
2. Crea `envio_por_ciudad` con los pedidos entregados: por `ciudad`, el número de pedidos (`n_pedidos`) y la suma de `costo_envio` (`ingreso_envios`, redondeada a 2). Ordena de mayor a menor `ingreso_envios`.

In [ ]:
tarifas_envio = spark.createDataFrame(
    [(0.0, 2.0, 3.0), (2.0, 5.0, 5.0), (5.0, 8.0, 7.5), (8.0, 100.0, 10.0)],
    "km_desde DOUBLE, km_hasta DOUBLE, costo_envio DOUBLE",
)


display(envio_por_ciudad)

In [ ]:
# comprobación
assert pedidos_envio.count() == pedidos_enriquecidos.count(), "Cada pedido debe caer en exactamente un tramo"
assert "km_desde" not in pedidos_envio.columns and "costo_envio" in pedidos_envio.columns
assert pedidos_envio.filter((F.col("distancia_km") < 2) & (F.col("costo_envio") != 3.0)).count() == 0
assert pedidos_envio.filter((F.col("distancia_km") == 5.0) & (F.col("costo_envio") != 7.5)).count() == 0, "5 km ya es del tramo [5, 8)"
assert envio_por_ciudad.columns == ["ciudad", "n_pedidos", "ingreso_envios"] and envio_por_ciudad.count() == 3
ok()

## Módulo 6. Agregaciones con más nivel

Además de `count`, `sum`, `avg`, `min` y `max`:

```python
df.groupBy("grupo").agg(
    F.countDistinct("cliente").alias("clientes"),               # valores distintos
    F.median("monto").alias("mediana"),                         # mediana
    F.avg(F.when(F.col("estado") == "cancelado", 1).otherwise(0)).alias("tasa"),   # proporción que cumple algo
    F.avg(F.when(F.col("estado") == "entregado", F.col("monto"))).alias("media_entregados"),  # promedio de un subconjunto
    F.array_sort(F.collect_set("categoria")).alias("categorias"),   # junta los valores distintos en un array
)
```

El truco de `F.when(...)` sin `otherwise` dentro de una agregación es muy útil: las filas que no cumplen quedan en `null`, y `avg`, `sum`, `median` y compañía ignoran los nulos. Así puedes calcular varias métricas sobre subconjuntos distintos en un solo `groupBy`.

`pivot` convierte los valores de una columna en columnas:

```python
df.groupBy("mes").pivot("metodo_pago", ["yape", "plin"]).agg(F.sum("total"))
```
Si le pasas la lista de valores, Spark no tiene que recorrer los datos para averiguarlos y las columnas salen en ese orden.

### Ejercicio 6.1: KPIs por ciudad
Crea `kpis_ciudad` a partir de `pedidos_enriquecidos`, agrupando por `ciudad` (la del restaurante), con:
- `n_pedidos`: todos los pedidos
- `clientes_unicos`: clientes distintos
- `tasa_cancelacion`: porcentaje (0 a 100) de pedidos con estado `"cancelado"`, redondeado a 2
- `ticket_medio`: promedio de `total` solo de los pedidos entregados, redondeado a 2
- `ticket_mediano`: mediana de `total` solo de los pedidos entregados, redondeada a 2

Ordena por `n_pedidos` de mayor a menor. Todo en un solo `groupBy`.

In [ ]:
display(kpis_ciudad)

In [ ]:
# comprobación
assert kpis_ciudad.columns == ["ciudad", "n_pedidos", "clientes_unicos", "tasa_cancelacion", "ticket_medio", "ticket_mediano"]
k = {r["ciudad"]: r for r in kpis_ciudad.collect()}
assert set(k) == {"Lima", "Arequipa", "Trujillo"}
assert sum(r["n_pedidos"] for r in k.values()) == pedidos_enriquecidos.count()
lima = pedidos_enriquecidos.filter("ciudad = 'Lima'")
assert abs(k["Lima"]["ticket_medio"] - lima.filter("estado = 'entregado'").agg(F.avg("total")).first()[0]) < 0.01, \
    "ticket_medio debe considerar solo los pedidos entregados"
assert abs(k["Lima"]["tasa_cancelacion"] - 100 * lima.filter("estado = 'cancelado'").count() / lima.count()) < 0.01, \
    "tasa_cancelacion es un porcentaje entre 0 y 100"
ok()

### Ejercicio 6.2: ingresos por mes y método de pago (`pivot`)
Crea `ventas_mes_pago` con los pedidos entregados: una fila por `anio_mes` y una columna por método de pago (`yape`, `plin`, `tarjeta`, `efectivo`, en ese orden) con la suma de `total` redondeada a 2. Ordena por `anio_mes`.

Después del `display`, prueba armar un gráfico de barras apiladas desde el botón + > Visualización.

In [ ]:
display(ventas_mes_pago)

In [ ]:
# comprobación
assert ventas_mes_pago.columns == ["anio_mes", "yape", "plin", "tarjeta", "efectivo"], f"Columnas: {ventas_mes_pago.columns}"
filas = ventas_mes_pago.collect()
assert len(filas) == 6 and [r["anio_mes"] for r in filas] == sorted(r["anio_mes"] for r in filas)
suma = sum(r[c] or 0 for r in filas for c in ["yape", "plin", "tarjeta", "efectivo"])
total = pedidos_enriquecidos.filter("estado = 'entregado'").agg(F.sum("total")).first()[0]
assert abs(suma - total) < 1, "La suma de todas las celdas debería ser el total vendido (solo entregados)"
ok()

### Ejercicio 6.3: categorías que prueba cada cliente
Crea `categorias_cliente` con una fila por `id_cliente` y estas columnas:
- `categorias`: array ordenado con las categorías distintas que pidió (usa `collect_set` y `array_sort`)
- `n_categorias`: cuántas son (`F.size`)

Luego crea `exploradores` con los clientes que probaron 5 categorías o más, y guarda cuántos son en `n_exploradores`.

In [ ]:
print(f"{n_exploradores} clientes probaron 5 categorías o más")
display(exploradores.orderBy(F.desc("n_categorias")).limit(10))

In [ ]:
# comprobación
assert categorias_cliente.columns == ["id_cliente", "categorias", "n_categorias"]
assert categorias_cliente.count() == pedidos_enriquecidos.select("id_cliente").distinct().count()
assert categorias_cliente.agg(F.max("n_categorias")).first()[0] <= 8, "Usa collect_set (sin repetidos), no collect_list"
fila = categorias_cliente.filter(F.col("n_categorias") >= 3).first()
assert list(fila["categorias"]) == sorted(fila["categorias"]), "Ordena el array con array_sort"
assert n_exploradores == categorias_cliente.filter("n_categorias >= 5").count()
ok()

### Ejercicio 6.4: los 10 productos que más venden
Con `items_pedido` (solo pedidos entregados), crea `top_productos` agrupando por `producto`:
- `unidades`: suma de `cantidad`
- `ingresos`: suma de `subtotal_item`, redondeada a 2
- `n_restaurantes`: en cuántos restaurantes distintos se vendió

Quédate con los 10 de mayores ingresos, de mayor a menor.

In [ ]:
display(top_productos)

In [ ]:
# comprobación
filas = top_productos.collect()
assert top_productos.columns == ["producto", "unidades", "ingresos", "n_restaurantes"]
assert len(filas) == 10 and all(filas[i]["ingresos"] >= filas[i + 1]["ingresos"] for i in range(9))
maximo = (items_pedido.filter("estado = 'entregado'").groupBy("producto")
          .agg(F.round(F.sum("subtotal_item"), 2).alias("i")).agg(F.max("i")).first()[0])
assert filas[0]["ingresos"] == maximo, "¿Filtraste solo los entregados?"
ok()

## Módulo 7. Ventanas con más nivel

En la práctica 1 usamos `row_number`, `lag` y un acumulado. Ahora vamos a jugar con el marco de la ventana (qué filas mira cada cálculo):

| Ventana | Qué filas usa |
|---|---|
| `Window.partitionBy("g").orderBy("f").rowsBetween(Window.unboundedPreceding, Window.currentRow)` | desde la primera fila del grupo hasta la actual (acumulado) |
| `Window.orderBy("fecha").rowsBetween(-6, 0)` | la fila actual y las 6 anteriores (media móvil de 7 filas) |
| `Window.partitionBy("g")` (sin `orderBy`) | todo el grupo: sirve para totales por grupo y porcentajes |
| `Window.partitionBy()` (vacía) | toda la tabla |

Y las funciones de ranking se diferencian en cómo tratan los empates:

In [ ]:
notas = spark.createDataFrame([("Ana", 18), ("Luis", 17), ("Marta", 17), ("Jorge", 15), ("Sofía", 12)], "alumno STRING, nota INT")
w = Window.orderBy(F.desc("nota"))

display(notas.select(
    "alumno", "nota",
    F.row_number().over(w).alias("row_number"),   # 1, 2, 3, 4, 5: no repite
    F.rank().over(w).alias("rank"),               # 1, 2, 2, 4, 5: repite y salta
    F.dense_rank().over(w).alias("dense_rank"),   # 1, 2, 2, 3, 4: repite sin saltar
    F.ntile(2).over(w).alias("ntile_2"),          # parte las filas en 2 grupos de tamaño parecido
))

Spark avisa con un warning cuando una ventana no tiene `partitionBy`, porque tiene que llevar todas las filas a una sola partición. Con tablas chicas como estas no es problema.

### Ejercicio 7.1: top 3 restaurantes por ciudad
1. Crea `ventas_restaurante` con los pedidos entregados de `pedidos_enriquecidos`, agrupando por `ciudad`, `id_restaurante` y `restaurante`, con `ingresos` = suma de `total` redondeada a 2. Agrupa también por el id porque hay cadenas con el mismo nombre en distintas ciudades.
2. Crea `top3_restaurantes` con los 3 restaurantes de más ingresos de cada ciudad. Usa `dense_rank` en una columna `puesto`. Columnas: `ciudad`, `id_restaurante`, `restaurante`, `ingresos`, `puesto`, ordenadas por ciudad y puesto.

In [ ]:
display(top3_restaurantes)

In [ ]:
# comprobación
assert top3_restaurantes.columns == ["ciudad", "id_restaurante", "restaurante", "ingresos", "puesto"]
assert top3_restaurantes.agg(F.max("puesto")).first()[0] == 3
assert top3_restaurantes.select("ciudad").distinct().count() == 3
maximos = ventas_restaurante.groupBy("ciudad").agg(F.max("ingresos").alias("maximo"))
assert top3_restaurantes.filter("puesto = 1").join(maximos, "ciudad").filter(F.col("ingresos") != F.col("maximo")).count() == 0, \
    "El puesto 1 debe ser el de mayores ingresos de su ciudad"
ok()

### Ejercicio 7.2: participación de cada restaurante en su ciudad
A partir de `ventas_restaurante`, crea `participacion` agregando `pct_ciudad`: el porcentaje que representan los ingresos del restaurante sobre el total de su ciudad, redondeado a 2.

Pista: el total de la ciudad es `F.sum("ingresos").over(Window.partitionBy("ciudad"))`. No hace falta un groupBy ni un join.

In [ ]:
display(participacion.orderBy("ciudad", F.desc("pct_ciudad")))

In [ ]:
# comprobación
assert participacion.count() == ventas_restaurante.count(), "Una ventana no colapsa filas"
for r in participacion.groupBy("ciudad").agg(F.sum("pct_ciudad").alias("s")).collect():
    assert abs(r["s"] - 100) < 0.2, f"En {r['ciudad']} los porcentajes suman {r['s']}, deberían sumar 100"
ok()

### Ejercicio 7.3: tendencia diaria
`diario` ya tiene los pedidos entregados por día. Crea `tendencia` agregando:
- `media_movil_7d`: promedio de `n_pedidos` del día y los 6 anteriores, redondeado a 1
- `var_semanal`: `n_pedidos` menos los `n_pedidos` de 7 días antes (con `F.lag(..., 7)`)

Ordena por fecha. Después prueba un gráfico de líneas con `fecha` en el eje X y `n_pedidos` y `media_movil_7d` en el eje Y: la media móvil suaviza los picos de los fines de semana.

In [ ]:
diario = (silver_pedidos
    .filter(F.col("estado") == "entregado")
    .groupBy("fecha")
    .agg(F.count("*").alias("n_pedidos"), F.round(F.sum("total"), 2).alias("ingresos"))
)


display(tendencia)

In [ ]:
# comprobación
f = tendencia.orderBy("fecha").collect()
assert abs(f[0]["media_movil_7d"] - f[0]["n_pedidos"]) < 0.051, "El primer día solo se promedia a sí mismo"
assert abs(f[10]["media_movil_7d"] - sum(x["n_pedidos"] for x in f[4:11]) / 7) < 0.051, \
    "La media móvil debe usar el día actual y los 6 anteriores"
assert all(x["var_semanal"] is None for x in f[:7]), "Los primeros 7 días no tienen semana anterior"
assert f[7]["var_semanal"] == f[7]["n_pedidos"] - f[0]["n_pedidos"]
ok()

### Ejercicio 7.4: cada cuántos días pide un cliente
1. Crea `con_anterior` a partir de los pedidos entregados de `silver_pedidos`, agregando `dias_desde_anterior`: días entre la `fecha` del pedido y la del pedido anterior del mismo cliente (`F.datediff` + `F.lag`). Ordena la ventana por `fecha_hora` y, para desempatar, por `id_pedido`.
2. Crea `frecuencia` con una fila por cliente: `id_cliente`, `n_pedidos` y `promedio_dias` (promedio de `dias_desde_anterior`, redondeado a 1). Deja solo a los clientes con 2 pedidos o más.

In [ ]:
display(frecuencia.orderBy("promedio_dias").limit(10))

In [ ]:
# comprobación
clientes = silver_pedidos.filter("estado = 'entregado'").select("id_cliente").distinct().count()
assert con_anterior.filter(F.col("dias_desde_anterior").isNull()).count() == clientes, \
    "Solo el primer pedido de cada cliente debería tener dias_desde_anterior nulo: ¿particionaste por cliente?"
assert con_anterior.filter(F.col("dias_desde_anterior") < 0).count() == 0, "Hay días negativos: revisa el orden de la ventana"
assert frecuencia.columns == ["id_cliente", "n_pedidos", "promedio_dias"]
assert frecuencia.filter(F.col("promedio_dias").isNull() | (F.col("n_pedidos") < 2)).count() == 0
ok()

### Ejercicio 7.5: cuartiles de gasto (`ntile`)
¿Qué parte de los ingresos viene del 25 % de clientes que más gasta?

1. Crea `gasto_cliente` con los pedidos entregados de clientes registrados (`plan` no nulo en `pedidos_enriquecidos`): por `id_cliente`, `gasto_total` (suma de `total`, redondeada a 2) y `cuartil` = `ntile(4)` ordenando por `gasto_total` de mayor a menor (el cuartil 1 es el que más gasta).
2. Crea `resumen_cuartiles` con una fila por `cuartil`: `n_clientes`, `gasto_medio` (redondeado a 2) y `pct_ingresos` (porcentaje del gasto total que aporta ese cuartil, redondeado a 2). Ordena por cuartil.

Pista para `pct_ingresos`: las ventanas se calculan después del `groupBy`, así que se pueden aplicar sobre un agregado: `F.sum(F.sum("gasto_total")).over(Window.partitionBy())` es el total de toda la tabla.

In [ ]:
display(resumen_cuartiles)

In [ ]:
# comprobación
f = resumen_cuartiles.collect()
assert resumen_cuartiles.columns == ["cuartil", "n_clientes", "gasto_medio", "pct_ingresos"] and len(f) == 4
assert gasto_cliente.count() == pedidos_enriquecidos.filter("estado = 'entregado' AND plan IS NOT NULL").select("id_cliente").distinct().count()
assert max(r["n_clientes"] for r in f) - min(r["n_clientes"] for r in f) <= 1, "ntile reparte en grupos del mismo tamaño"
assert all(f[i]["gasto_medio"] > f[i + 1]["gasto_medio"] for i in range(3)), "El cuartil 1 debe ser el de mayor gasto"
assert abs(sum(r["pct_ingresos"] for r in f) - 100) < 0.1, "Los porcentajes deben sumar 100"
ok(f"Correcto. El 25 % de clientes que más gasta aporta el {f[0]['pct_ingresos']} % de los ingresos")

## Módulo 8. Spark SQL con más nivel

Tres cosas nuevas:

1. CTE (`WITH`): pones nombre a una subconsulta y la usas como si fuera una tabla. Hace que las consultas largas se lean de arriba hacia abajo.
   ```sql
   WITH ventas AS (SELECT ... GROUP BY ...),
        ranking AS (SELECT *, ROW_NUMBER() OVER (...) AS pos FROM ventas)
   SELECT * FROM ranking WHERE pos = 1
   ```
2. Consultas con parámetros: `spark.sql("... WHERE ciudad = :ciudad", args={"ciudad": "Lima"})`. Nunca armes SQL pegando con f-strings un valor que escribió un usuario: alguien podría mandar `Lima' OR '1'='1` y cambiar el sentido de la consulta (inyección SQL). Con parámetros, el valor siempre se trata como dato.
3. Las tablas que guardaste (`silver_pedidos`, etc.) se pueden consultar directo por su nombre, sin crear vistas.

### Ejercicio 8.1: vista temporal + CTE + ventana en SQL
Registra `pedidos_enriquecidos` como la vista temporal `pedidos_enriquecidos`. Luego, con `spark.sql` y un CTE, crea `top_categoria_mes`: para cada `anio_mes`, la categoría con más ingresos (suma de `total` de los pedidos entregados, redondeada a 2). Columnas: `anio_mes`, `categoria`, `ingresos`, ordenado por `anio_mes`.

Cuando tengas el resultado, míralo con calma: ¿la categoría líder es la misma todo el semestre? (Pista: en Perú, de enero a marzo es verano.)

In [ ]:
display(top_categoria_mes)

In [ ]:
# comprobación
assert top_categoria_mes.columns == ["anio_mes", "categoria", "ingresos"] and top_categoria_mes.count() == 6
esperado = (pedidos_enriquecidos.filter("estado = 'entregado'")
            .groupBy("anio_mes", "categoria").agg(F.round(F.sum("total"), 2).alias("ingresos"))
            .groupBy("anio_mes").agg(F.max("ingresos").alias("ingresos")))
assert {(r["anio_mes"], r["ingresos"]) for r in esperado.collect()} == \
       {(r["anio_mes"], r["ingresos"]) for r in top_categoria_mes.collect()}
ok()

### Ejercicio 8.2: consulta con parámetros
Escribe la función `resumen_ciudad_mes(ciudad, anio_mes)`, que devuelva un DataFrame de una fila con:
- `n_pedidos`: todos los pedidos de esa ciudad y mes
- `ingresos`: suma de `total` solo de los entregados, redondeada a 2 (pista: `SUM(CASE WHEN ... THEN total END)`)
- `calificacion_media`: promedio de `calificacion`, redondeado a 1

Consulta la vista `pedidos_enriquecidos` con `spark.sql` usando los marcadores `:ciudad` y `:anio_mes` y el argumento `args`.

In [ ]:
def resumen_ciudad_mes(ciudad, anio_mes):
    # tu código aquí
    pass


display(resumen_ciudad_mes("Arequipa", "2025-03"))

In [ ]:
# comprobación
for ciudad, mes in [("Arequipa", "2025-03"), ("Lima", "2025-06")]:
    r = resumen_ciudad_mes(ciudad, mes).first()
    filtro = (F.col("ciudad") == ciudad) & (F.col("anio_mes") == mes)
    assert r["n_pedidos"] == pedidos_enriquecidos.filter(filtro).count(), f"n_pedidos no cuadra para {ciudad} {mes}"
    esperado = pedidos_enriquecidos.filter(filtro & (F.col("estado") == "entregado")).agg(F.sum("total")).first()[0]
    assert abs(r["ingresos"] - esperado) < 0.01, f"ingresos no cuadra para {ciudad} {mes}"
ok()

### Ejercicio 8.3: celda `%sql` sobre una tabla
En la celda de abajo, consulta directamente la tabla `silver_pedidos` (solo pedidos entregados) y devuelve, para cada `hora`:
- `franja`: `"almuerzo"` si la hora está entre 12 y 15, `"cena"` si está entre 19 y 22, `"otra"` en los demás casos (`CASE WHEN`)
- `n_pedidos`
- `minutos_medios`: promedio de `minutos_entrega`, redondeado a 1

Ordena por hora y arma un gráfico de barras con el resultado.

Dato: Databricks SQL tiene además la cláusula `QUALIFY`, que filtra por el resultado de una ventana sin necesidad de subconsulta (por ejemplo `QUALIFY ROW_NUMBER() OVER (...) = 1`). No es parte del Spark estándar, pero en Databricks funciona y ahorra el CTE del 8.1.

In [ ]:
%sql
-- consulta sobre silver_pedidos

## Módulo 9. Delta Lake: MERGE y time travel

Cada escritura en una tabla Delta crea una versión nueva. Eso permite:
- ver el historial de cambios: `DESCRIBE HISTORY tabla`
- leer una versión anterior (time travel): `SELECT * FROM tabla VERSION AS OF 3`, o en Python `spark.read.option("versionAsOf", 3).table("tabla")`
- volver atrás si algo salió mal: `RESTORE TABLE tabla TO VERSION AS OF 3`
- actualizar e insertar en una sola operación con `MERGE INTO` (lo que se suele llamar upsert):

```sql
MERGE INTO destino AS d
USING origen AS o
ON d.id = o.id
WHEN MATCHED THEN UPDATE SET *        -- si ya existe, se actualiza
WHEN NOT MATCHED THEN INSERT *        -- si no existe, se inserta
```
`UPDATE SET *` e `INSERT *` funcionan cuando origen y destino tienen las mismas columnas. Si no, se listan: `UPDATE SET d.comision_pct = o.comision_pct`.

Llegó el archivo de julio con cambios de restaurantes: algunos cambiaron su comisión y hay dos restaurantes nuevos. En lugar de reescribir toda la tabla, hacemos un MERGE.

### Ejercicio 9.1: preparar los cambios
1. Lee `ruta_cambios` con el mismo formato que `restaurantes.csv` (encabezado, `;` y `ESQUEMA_RESTAURANTES`) y límpialo con tu función `limpiar_restaurantes` usando `transform`. Guárdalo en `cambios_restaurantes`.
2. Guarda en `version_antes` el número de la versión actual de `silver_restaurantes`: es el máximo de la columna `version` de `DESCRIBE HISTORY silver_restaurantes`.

In [ ]:
ruta_cambios = RUTA_RAW + "/actualizaciones/restaurantes_2025_07.csv"


display(cambios_restaurantes)
print("Versión actual de silver_restaurantes:", version_antes)

In [ ]:
# comprobación
assert cambios_restaurantes.count() == 6
assert cambios_restaurantes.columns == silver_restaurantes.columns, "Usa limpiar_restaurantes para tener las mismas columnas que la tabla"
assert isinstance(version_antes, int) and version_antes >= 0
ok()

### Ejercicio 9.2: `MERGE INTO`
Registra `cambios_restaurantes` como la vista temporal `cambios_restaurantes` y haz un `MERGE INTO silver_restaurantes` que actualice los restaurantes que ya existen (por `id_restaurante`) e inserte los nuevos. Muestra el resultado del `spark.sql` (trae cuántas filas se actualizaron e insertaron) y vuelve a leer la tabla en `silver_restaurantes`.

In [ ]:
display(silver_restaurantes.orderBy("id_restaurante"))

In [ ]:
# comprobación
assert silver_restaurantes.count() == 42, "Deberían quedar 42 restaurantes (40 + 2 nuevos)"
nuevos = {r["id_restaurante"]: r["comision_pct"] for r in cambios_restaurantes.collect()}
actuales = {r["id_restaurante"]: r["comision_pct"] for r in silver_restaurantes.collect()}
assert all(actuales[k] == v for k, v in nuevos.items()), "Las comisiones no se actualizaron"
assert spark.sql("DESCRIBE HISTORY silver_restaurantes").agg(F.max("version")).first()[0] > version_antes
ok()

El MERGE quedó registrado en el historial. En `operationMetrics` puedes ver cuántas filas se actualizaron (`numTargetRowsUpdated`) y cuántas se insertaron (`numTargetRowsInserted`):

In [ ]:
display(spark.sql("DESCRIBE HISTORY silver_restaurantes").select("version", "timestamp", "operation", "operationMetrics"))

### Ejercicio 9.3: time travel
Lee la versión anterior al MERGE con `VERSION AS OF version_antes` y crea `comparacion` con los restaurantes cuya comisión cambió. Columnas: `id_restaurante`, `nombre` (el de la versión anterior), `comision_antes` y `comision_despues`, ordenado por `id_restaurante`.

Pista: como vas a unir la tabla consigo misma (dos versiones), renombra `comision_pct` en cada lado antes del join.

In [ ]:
display(comparacion)

In [ ]:
# comprobación
assert comparacion.columns == ["id_restaurante", "nombre", "comision_antes", "comision_despues"]
assert spark.sql(f"SELECT * FROM silver_restaurantes VERSION AS OF {version_antes}").count() == 40, \
    "version_antes debería ser la versión con 40 restaurantes. ¿Volviste a ejecutar el 9.1 después del MERGE?"
assert comparacion.count() == 4, f"Deberían ser 4 restaurantes con comisión distinta, hay {comparacion.count()}"
ok()

## Módulo 10. Gold y exportación

La capa gold tiene tablas pensadas para quien consume los datos: dashboards, reportes o el equipo comercial. Además vamos a exportar archivos a un volumen:

- Parquet particionado: `df.write.partitionBy("anio_mes").parquet(ruta)` crea una carpeta por valor (`anio_mes=2025-01/`, `anio_mes=2025-02/`...). Al leer con un filtro sobre esa columna, Spark solo abre las carpetas que necesita (partition pruning).
- Un CSV en un solo archivo: por defecto Spark escribe un archivo por partición. Con `coalesce(1)` se junta todo en una partición y sale un único `part-*.csv`. Úsalo solo con resultados chicos, porque obliga a que una sola tarea escriba todo.

### Ejercicio 10.1: tabla gold de ventas diarias
Crea `gold_ventas_diarias_df` con los pedidos entregados de `pedidos_enriquecidos`, agrupando por `fecha` y `ciudad`:
- `n_pedidos`
- `ingresos`: suma de `total`, redondeada a 2
- `ticket_medio`: promedio de `total`, redondeado a 2
- `comision_chaskifood`: suma de `total * comision_pct / 100`, redondeada a 2 (lo que gana ChaskiFood)

Ordena por fecha y ciudad, guárdalo como la tabla `gold_ventas_diarias` y léela en `gold_ventas_diarias`.

Un detalle para pensar: como `silver_restaurantes` ya tiene las comisiones de julio, este cálculo aplica la comisión nueva también a pedidos de meses anteriores. Para hacerlo bien habría que guardar el historial de comisiones con sus fechas de vigencia (lo que se conoce como dimensión de cambio lento, SCD tipo 2). Lo dejamos así para no complicar el ejercicio.

In [ ]:
display(gold_ventas_diarias.limit(10))

In [ ]:
# comprobación
assert gold_ventas_diarias.columns == ["fecha", "ciudad", "n_pedidos", "ingresos", "ticket_medio", "comision_chaskifood"]
assert gold_ventas_diarias.groupBy("fecha", "ciudad").count().filter("count > 1").count() == 0, "Debe haber una fila por fecha y ciudad"
assert gold_ventas_diarias.agg(F.sum("n_pedidos")).first()[0] == pedidos_enriquecidos.filter("estado = 'entregado'").count()
assert gold_ventas_diarias.filter(F.col("comision_chaskifood") >= F.col("ingresos")).count() == 0, "La comisión es un porcentaje de los ingresos"
ok()

### Ejercicio 10.2: exportar a Parquet particionado
1. Escribe `pedidos_enriquecidos` en formato Parquet en `ruta_parquet`, particionado por `anio_mes`, en modo overwrite.
2. Crea `pedidos_marzo` leyendo esa carpeta y filtrando `anio_mes == "2025-03"`. Fíjate que `anio_mes` vuelve a aparecer como columna aunque no está dentro de los archivos: Spark la saca del nombre de la carpeta.

In [ ]:
ruta_parquet = RUTA_VOLUMEN + "/export/pedidos_parquet"


display(dbutils.fs.ls(ruta_parquet))
print(pedidos_marzo.count())

In [ ]:
# comprobación
carpetas = sorted(f.name for f in dbutils.fs.ls(ruta_parquet) if f.name.startswith("anio_mes="))
assert len(carpetas) == 6, f"Esperaba 6 carpetas anio_mes=..., hay {carpetas}"
assert pedidos_marzo.count() == pedidos_enriquecidos.filter(F.col("anio_mes") == "2025-03").count()
ok()

En el plan de `pedidos_marzo` aparece el filtro como `PartitionFilters`: Spark sabe que solo tiene que leer la carpeta `anio_mes=2025-03`.

In [ ]:
pedidos_marzo.explain()

### Ejercicio 10.3: reporte CSV en un solo archivo
Escribe `top3_restaurantes` (ejercicio 7.1) como CSV con encabezado en `ruta_reporte`, en modo overwrite, usando `coalesce(1)` para que quede en un único archivo `part-*`.

In [ ]:
ruta_reporte = RUTA_VOLUMEN + "/export/top3_restaurantes"


display(dbutils.fs.ls(ruta_reporte))

In [ ]:
# comprobación
partes = [f.name for f in dbutils.fs.ls(ruta_reporte) if f.name.startswith("part-")]
assert len(partes) == 1, f"Debería haber un solo archivo part-*, hay {len(partes)}"
assert spark.read.option("header", True).csv(ruta_reporte).count() == top3_restaurantes.count()
ok("Correcto. Desde Catalog > workspace > chaskifood > archivos puedes descargar el CSV")

## Desafío final

Dos preguntas que el equipo de negocio de ChaskiFood hace siempre. No hay plantilla: decide tú los pasos.

### Desafío A: retención por cohortes
De los clientes que hicieron su primer pedido en cada mes, ¿qué porcentaje volvió a pedir 1, 2, 3... meses después?

Usa solo pedidos entregados de clientes registrados (`plan` no nulo en `pedidos_enriquecidos`). El resultado, `retencion`, debe tener:

| columna | significado |
|---|---|
| `mes_cohorte` | mes del primer pedido del cliente (`"2025-01"`, ...) |
| `clientes` | cuántos clientes empezaron ese mes |
| `m0` ... `m5` | porcentaje de esos clientes que pidió 0, 1, ..., 5 meses después de su primer mes (redondeado a 1). `m0` siempre es 100. Donde todavía no hay datos (por ejemplo, `m1` de la cohorte de junio) queda `null` |

Ordena por `mes_cohorte`.

Un camino posible:
1. `mes_cohorte` = el `anio_mes` mínimo de cada cliente (una ventana `partitionBy("id_cliente")` sin `orderBy`).
2. `meses_desde` = meses entre `anio_mes` y `mes_cohorte`: `F.months_between(F.to_date("anio_mes", "yyyy-MM"), F.to_date("mes_cohorte", "yyyy-MM")).cast("int")`.
3. Clientes distintos por `mes_cohorte` y `meses_desde`, con un `pivot` sobre `meses_desde` (valores 0 a 5). Ojo: las columnas del pivot se llaman `"0"`, `"1"`... (texto).
4. Divide cada columna por la columna `"0"`.

In [ ]:
display(retencion)

In [ ]:
# comprobación
filas = retencion.collect()
assert retencion.columns == ["mes_cohorte", "clientes", "m0", "m1", "m2", "m3", "m4", "m5"], f"Columnas: {retencion.columns}"
assert [r["mes_cohorte"] for r in filas] == ["2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"]
assert all(r["m0"] == 100.0 for r in filas), "m0 debe ser 100 en todas las cohortes"
registrados = pedidos_enriquecidos.filter("estado = 'entregado' AND plan IS NOT NULL").select("id_cliente").distinct().count()
assert sum(r["clientes"] for r in filas) == registrados, "Cada cliente debe pertenecer a una sola cohorte"
assert filas[-1]["m1"] is None and filas[0]["m5"] is not None
assert all(0 < r["m1"] < 100 for r in filas[:-1])
ok("Desafío A listo")

### Desafío B: segmentación de clientes
Marketing quiere una tabla para sus campañas. Crea `perfil_clientes` con una fila por cliente registrado que tenga al menos un pedido entregado, y estas columnas en este orden:

| columna | cálculo |
|---|---|
| `id_cliente` | |
| `n_pedidos` | pedidos entregados |
| `gasto_total` | suma de `total`, redondeada a 2 |
| `ultima_compra` | `fecha` del último pedido entregado |
| `dias_sin_comprar` | días entre `ultima_compra` y `FECHA_CORTE` (30 de junio de 2025) |
| `cuartil_gasto` | `ntile(4)` por `gasto_total` de mayor a menor |
| `segmento` | ver reglas abajo |

Reglas de `segmento`, en este orden:
- `"VIP activo"`: cuartil 1 y `dias_sin_comprar` <= 30
- `"VIP en riesgo"`: cuartil 1 y más de 30 días sin comprar
- `"Dormido"`: más de 60 días sin comprar
- `"Regular"`: el resto

Después:
1. Guárdalo como la tabla `gold_segmentos_clientes` y léela en `gold_segmentos`.
2. Crea `vip_en_riesgo` para que marketing los llame: los clientes del segmento `"VIP en riesgo"` con `id_cliente`, `nombre`, `email`, `telefono` (de `silver_clientes`), `gasto_total` y `dias_sin_comprar`, ordenados de mayor a menor gasto.

In [ ]:
FECHA_CORTE = date(2025, 6, 30)


display(gold_segmentos.groupBy("segmento").agg(F.count("*").alias("n_clientes"), F.round(F.avg("gasto_total"), 2).alias("gasto_medio")))
display(vip_en_riesgo)

In [ ]:
# comprobación
assert gold_segmentos.columns == ["id_cliente", "n_pedidos", "gasto_total", "ultima_compra", "dias_sin_comprar", "cuartil_gasto", "segmento"], \
    f"Columnas: {gold_segmentos.columns}"
registrados = pedidos_enriquecidos.filter("estado = 'entregado' AND plan IS NOT NULL").select("id_cliente").distinct().count()
assert gold_segmentos.count() == registrados
segmentos = {r["segmento"] for r in gold_segmentos.select("segmento").distinct().collect()}
assert segmentos == {"VIP activo", "VIP en riesgo", "Dormido", "Regular"}, f"Segmentos: {segmentos}"
assert gold_segmentos.filter(F.col("segmento").startswith("VIP")).count() == gold_segmentos.filter("cuartil_gasto = 1").count(), \
    "Todos los del cuartil 1 son VIP (activos o en riesgo)"
assert gold_segmentos.filter((F.col("segmento") == "Dormido") & (F.col("dias_sin_comprar") <= 60)).count() == 0
assert vip_en_riesgo.columns == ["id_cliente", "nombre", "email", "telefono", "gasto_total", "dias_sin_comprar"]
assert vip_en_riesgo.count() == gold_segmentos.filter("segmento = 'VIP en riesgo'").count()
ok("Desafío B listo. Terminaste la práctica 2")

## Limpieza (opcional)
Para borrar todo lo que creó esta práctica (tablas, volumen y archivos), descomenta la línea y ejecútala:

In [ ]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOGO}.{ESQUEMA} CASCADE")

## Para seguir
- Funciones de PySpark (la lista completa, incluidas las de arrays y fechas): https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html
- Delta Lake en Databricks (MERGE, time travel, historial): https://docs.databricks.com/aws/en/delta/
- En este notebook leímos todos los archivos cada vez. Cuando los archivos llegan continuamente, Databricks usa Auto Loader, que procesa solo los archivos nuevos: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/
- El pipeline bronze, silver y gold que armamos a mano es justo lo que automatizan los pipelines declarativos de Lakeflow: https://docs.databricks.com/aws/en/dlt/